# 📡 Lab 05 · Your GPS jumps. Should you jump with it?

**World Models course · Part A: Lecture 4 (HW1 Part A) · Part B: Lectures 15–16** &nbsp;|&nbsp; ⏱ about 60 min &nbsp;|&nbsp; 💻 CPU only

**Part A: Filtering.** A car drives down a road. Its GPS is noisy and cuts out in tunnels. You will build the algorithm that runs in every phone, drone and self-driving car to fuse a **prediction** (*"I was here, moving this fast"*) with a **measurement** (*"GPS says I'm there"*): the **Kalman filter**.

**Part B: Cameras.** A photo flattens 3D into 2D. You will project 3D points into pixels, see why a nearby small object and a distant big object can look identical, and recover 3D using depth. These are the foundations of NeRF, 3D Gaussian Splatting and every robot's camera.

🧩 challenges · 🔮 predictions · 🎛️ playgrounds. Blank or wrong answers never break the notebook.

In [ ]:
#@title 🔧 Step 0 · Run this cell first (click ▶). It loads the tools for this lab. { display-mode: "form" }
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

# ---------------------------------------------------------------------------
# Guided-lab helpers. You never need to edit this cell.
#  * ___            : a blank for you to fill in
#  * check(name, x) : checks your answer; if it is blank or wrong, it explains
#                     and hands back a working version so the notebook keeps going
#  * quiz(id)       : a clickable multiple-choice question
#  * playground(...) : sliders that re-run a function when you let go
# ---------------------------------------------------------------------------
import inspect, html as _html
import numpy as np
from IPython.display import display, HTML
import os
try:
    import ipywidgets as widgets
    _WIDGETS = not os.environ.get("GUIDE_NO_WIDGETS")
except Exception:
    _WIDGETS = False

class BlankNotFilled(Exception):
    pass

class _Blank:
    """The ___ placeholder. Any maths with it stops with a friendly message."""
    __array_ufunc__ = None
    def _stop(self, *args, **kwargs):
        raise BlankNotFilled("There is still a ___ blank to fill in.")
    __add__ = __radd__ = __sub__ = __rsub__ = __mul__ = __rmul__ = _stop
    __truediv__ = __rtruediv__ = __floordiv__ = __rfloordiv__ = _stop
    __pow__ = __rpow__ = __matmul__ = __rmatmul__ = __mod__ = __rmod__ = _stop
    __neg__ = __pos__ = __abs__ = __getitem__ = __call__ = __iter__ = _stop
    __lt__ = __le__ = __gt__ = __ge__ = __bool__ = __float__ = __int__ = __index__ = _stop
    __array__ = __len__ = _stop
    def __getattr__(self, name):
        if name.startswith('__'):
            raise AttributeError(name)
        raise BlankNotFilled("There is still a ___ blank to fill in.")
    def __repr__(self):
        return "___"

___ = _Blank()
CHALLENGES, QUIZZES = {}, {}
_solved, _quiz_score = {}, {}

_STYLE = {
    "ok":   ("#e8f6ee", "#1b7a4b", "✅"),
    "wait": ("#fff5e0", "#9a5b00", "🧩"),
    "bad":  ("#fdecea", "#b3261e", "❌"),
    "info": ("#eaf1fb", "#245eb5", "💡"),
}

def card(kind, title, body=""):
    bg, fg, icon = _STYLE[kind]
    display(HTML(
        f'<div style="background:{bg};border-left:5px solid {fg};padding:10px 14px;'
        f'border-radius:6px;margin:6px 0;color:#1d2530;font-size:14px;line-height:1.5">'
        f'<b style="color:{fg}">{icon} {title}</b><div>{body}</div></div>'))

def _as_numpy(x):
    if hasattr(x, "detach"):
        x = x.detach().cpu().numpy()
    if isinstance(x, (list, tuple)):
        return [_as_numpy(v) for v in x]
    return x

def _same(a, b, tol):
    a, b = _as_numpy(a), _as_numpy(b)
    if isinstance(a, list) or isinstance(b, list):
        return isinstance(a, list) and isinstance(b, list) and len(a) == len(b) and all(_same(x, y, tol) for x, y in zip(a, b))
    try:
        a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    except Exception:
        return a == b
    return a.shape == b.shape and np.allclose(a, b, atol=tol, rtol=tol)

def _has_blank(obj):
    if isinstance(obj, _Blank):
        return True
    if callable(obj):
        try:
            return "___" in inspect.getsource(obj)
        except Exception:
            return False
    return False

def check(name, answer):
    """Check a challenge. Returns your answer if it works, otherwise a working reference."""
    ch = CHALLENGES[name]
    ref = ch["reference"]
    title = ch.get("title", name)
    fallback = ("<br><i>For now the notebook will use a working version so every later cell still runs. "
                "Come back, fill it in, and re-run this cell.</i>")
    if _has_blank(answer):
        _solved.setdefault(name, False)
        card("wait", f"Challenge “{title}” is waiting for you", "Hint: " + ch["hint"] + fallback)
        return ref
    try:
        if "test" in ch:
            ok, message = ch["test"](answer)
        elif callable(ref):
            ok, message = True, ""
            for args in ch["cases"]:
                args = args if isinstance(args, tuple) else (args,)
                expected, got = ref(*args), answer(*args)
                if not _same(expected, got, ch.get("tol", 1e-6)):
                    ok = False
                    message = "For a test input your function gave a different result from the expected one."
                    break
        else:
            ok = _same(ref, answer, ch.get("tol", 1e-6))
            message = f"You entered <code>{_html.escape(repr(_as_numpy(answer)))}</code>."
    except BlankNotFilled:
        _solved.setdefault(name, False)
        card("wait", f"Challenge “{title}” still has a blank", "Hint: " + ch["hint"] + fallback)
        return ref
    except Exception as err:
        ok, message = False, f"Running your version raised <code>{_html.escape(type(err).__name__)}: {_html.escape(str(err))}</code>."
    if ok:
        _solved[name] = True
        card("ok", f"Challenge solved: {title}", ch.get("why", ""))
        return answer
    _solved[name] = False
    card("bad", f"Not quite yet: {title}", message + "<br>Hint: " + ch["hint"] + fallback)
    return ref

def quiz(qid):
    q = QUIZZES[qid]
    question = f'<div style="font-size:15px;margin:8px 0 4px"><b>{"🔮 Predict: " if q.get("predict") else "🤔 "}{q["q"]}</b></div>'
    if not _WIDGETS:
        options = "".join(f"<li>{_html.escape(o)}</li>" for o in q["options"])
        display(HTML(question + f"<ol type='A'>{options}</ol><details><summary>Answer</summary>"
                     f"{'ABCDEFG'[q['answer']]}. {q['explain']}</details>"))
        return
    out = widgets.Output()
    buttons = []
    def choose(i):
        def handler(_):
            _quiz_score.setdefault(qid, i == q["answer"])
            for j, b in enumerate(buttons):
                b.button_style = "success" if j == q["answer"] else ("danger" if j == i else "")
            with out:
                out.clear_output()
                if i == q["answer"]:
                    card("ok", "Yes!", q["explain"])
                else:
                    card("bad", "Not this one. Here is the reasoning:", q["explain"])
        return handler
    for i, option in enumerate(q["options"]):
        b = widgets.Button(description=f"{'ABCDEFG'[i]}. {option}", layout=widgets.Layout(width="auto", max_width="100%"))
        b.on_click(choose(i))
        buttons.append(b)
    display(HTML(question), widgets.VBox(buttons), out)

def playground(fn, **controls):
    """controls: name=(min, max, step, default) for sliders, or name=[option, ...] for a dropdown."""
    defaults, sliders = {}, {}
    for name, spec in controls.items():
        if isinstance(spec, list):
            defaults[name] = spec[0]
            if _WIDGETS:
                sliders[name] = widgets.Dropdown(options=spec, value=spec[0], description=name)
        else:
            lo, hi, step, value = spec
            defaults[name] = value
            if _WIDGETS:
                kind = widgets.IntSlider if all(isinstance(v, int) for v in spec) else widgets.FloatSlider
                sliders[name] = kind(min=lo, max=hi, step=step, value=value, description=name,
                                     continuous_update=False, style={"description_width": "initial"},
                                     layout=widgets.Layout(width="420px"))
    if _WIDGETS:
        ui = widgets.VBox(list(sliders.values()))
        out = widgets.interactive_output(fn, sliders)
        display(ui, out)
    else:
        fn(**defaults)

def progress_report():
    solved = sum(_solved.values()); total = len(CHALLENGES)
    right = sum(_quiz_score.values()); asked = len(_quiz_score)
    stars = "⭐" * solved + "☆" * (total - solved)
    body = f"Challenges solved yourself: <b>{solved} / {total}</b> {stars}<br>"
    body += f"Quiz questions right on the first click: <b>{right} / {asked}</b> (of {len(QUIZZES)} in this lab)"
    missing = [CHALLENGES[k].get('title', k) for k in CHALLENGES if not _solved.get(k)]
    if missing:
        body += "<br>Still worth a try: " + ", ".join(missing)
    card("info", "Your progress in this lab", body)

# ---- this lab's challenges and quizzes ----
CHALLENGES["blend"] = dict(title="Blend prediction and measurement",
    reference=lambda estimate, measurement, trust: estimate + trust * (measurement - estimate),
    cases=[(2.0, 5.0, 0.25), (np.array([1., 2.]), np.array([3., 0.]), 0.5)],
    hint="Start at your estimate and move a fraction <code>trust</code> of the way toward the measurement.",
    why="<code>measurement − estimate</code> is the <b>surprise</b> (the innovation). Every filter corrects its belief by some fraction of the surprise.")

def _ref_A(dt):
    return np.array([[1.0, dt], [0.0, 1.0]])
CHALLENGES["motion_model"] = dict(title="The motion model", reference=_ref_A, cases=[0.25, 1.0],
    hint="Row 1 says: new position = 1 × position + (?) × velocity. How far do you travel in <code>dt</code> seconds at speed v?",
    why="This matrix <b>A</b> is the filter's tiny world model: it predicts the next state from the current one.")

def _ref_gain(P, R):
    return P[:, 0] / (P[0, 0] + R)
CHALLENGES["gain"] = dict(title="How much to trust the GPS", reference=_ref_gain,
    cases=[(np.array([[4.0, 1.0], [1.0, 2.0]]), 1.0), (np.array([[0.1, 0.0], [0.0, 1.0]]), 9.0)],
    hint="Position uncertainty is <code>P[0, 0]</code>. Trust = my uncertainty ÷ (my uncertainty + GPS noise <code>R</code>). The velocity row reuses <code>P[:, 0]</code> in the numerator.",
    why="The <b>Kalman gain</b>: if you're unsure (big P) and the GPS is good (small R), trust the GPS; if you're sure and the GPS is bad, mostly ignore it.")

CHALLENGES["project"] = dict(title="Project a 3D point to a pixel",
    reference=lambda X, Y, Z, f, cx, cy: (f * X / Z + cx, f * Y / Z + cy),
    cases=[(1.0, -0.5, 4.0, 250.0, 160.0, 120.0), (np.array([0., 2.]), np.array([1., 1.]), np.array([2., 8.]), 100.0, 50.0, 40.0)],
    hint="Divide by depth (far things shrink), multiply by focal length <code>f</code>, then shift to the image centre: u = f·X/Z + cx.",
    why="The <b>pinhole camera</b> model. Dividing by Z is why parallel railway lines meet at the horizon.")

CHALLENGES["unproject"] = dict(title="Back to 3D using depth",
    reference=lambda u, v, Z, f, cx, cy: ((u - cx) * Z / f, (v - cy) * Z / f, Z),
    cases=[(222.5, 88.75, 4.0, 250.0, 160.0, 120.0)],
    hint="Undo the projection: subtract the centre, divide by f, multiply by the depth Z.",
    why="A depth camera (or a depth-predicting network) lets a robot turn every pixel back into a 3D point, making a <b>point cloud</b>.")

QUIZZES["tunnel"] = dict(predict=True, q="In a tunnel there is no GPS. What happens to the filter's uncertainty band?",
    options=["It stays the same", "It grows steadily until GPS returns", "It shrinks, because nothing noisy comes in"],
    answer=1, explain="With no measurements, the filter keeps predicting, and each prediction adds process noise Q. When GPS returns, one reading snaps the uncertainty back down.")
QUIZZES["smoother"] = dict(q="The smoother is more accurate than the filter. Can a self-driving car use it to steer right now?",
    options=["Yes, always use the more accurate one", "No: the smoother needs measurements from the future"],
    answer=1, explain="The smoother looks back with <b>hindsight</b>. It's perfect for labelling data offline or evaluating a filter, but not for real-time control. Never compare them as if they had the same information.")
QUIZZES["overconfident"] = dict(predict=True, q="The filter <b>assumes</b> the GPS is more accurate than it really is. The 95% band will…",
    options=["contain the truth more often than 95%", "contain the truth less often than 95%, so it is overconfident", "be unaffected"],
    answer=1, explain="Believing measurements are better than they are makes the band too narrow. A wrong noise model gives <b>miscalibrated</b> uncertainty, a common silent failure in robotics.")
QUIZZES["depth"] = dict(predict=True, q="A point at (X, Y, Z) and another at (2X, 2Y, 2Z) are photographed. Their pixels are…",
    options=["different: the far one is closer to the centre", "identical", "different: the far one is further from the centre"],
    answer=1, explain="u = f·(2X)/(2Z) + cx = f·X/Z + cx. The 2s cancel. Every point along the same ray lands on the same pixel, so a single image cannot tell depth. That is why we need stereo, motion, depth sensors or learned priors.")
print('✅ Setup complete. Scroll down and run the cells in order.')

---
# Part A · Filtering 🚗

## 1 · The road, the car and a noisy GPS
* **Hidden state:** position and velocity. The driver randomly speeds up and slows down.
* **Observation:** GPS reports position only, with ±1.5 m noise, **every 0.25 s**. There are two tunnels with no signal.

In [ ]:
T, dt = 240, 0.25
jerkiness, gps_noise = 0.6, 1.5         # process noise (driver) and measurement noise (GPS)

def simulate_drive(seed):
    rng = np.random.default_rng(seed)
    truth = np.zeros((T, 2)); truth[0] = [0.0, 8.0]           # start at 0 m, 8 m/s
    for t in range(1, T):
        accel = rng.normal(0, jerkiness)                        # the driver's random push on the pedal
        truth[t, 0] = truth[t - 1, 0] + truth[t - 1, 1] * dt + 0.5 * accel * dt**2   # position
        truth[t, 1] = truth[t - 1, 1] + accel * dt                                    # velocity
    gps = truth[:, 0] + rng.normal(0, gps_noise, T)            # noisy position readings
    has_gps = np.ones(T, bool); has_gps[70:110] = False; has_gps[170:185] = False    # two tunnels
    return truth, gps, has_gps

truth, gps, has_gps = simulate_drive(6280)
time = np.arange(T) * dt

def road_plot(ax, lo=0, hi=T):
    ax.plot(time[lo:hi], truth[lo:hi, 0], "k", lw=2, label="true position")
    ax.scatter(time[lo:hi][has_gps[lo:hi]], gps[lo:hi][has_gps[lo:hi]], s=6, c="tab:gray", alpha=0.6, label="GPS")
    for start, end in [(70, 110), (170, 185)]:
        ax.axvspan(time[start], time[end - 1], color="tab:purple", alpha=0.08)

# Subtract a steady 8 m/s drift so the wiggles are visible
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(time, truth[:, 0] - 8 * time, "k", lw=2, label="true position − 8 m/s × t")
ax.scatter(time[has_gps], gps[has_gps] - 8 * time[has_gps], s=6, c="tab:gray", label="GPS − 8 m/s × t")
ax.set(xlabel="seconds", ylabel="metres (drift removed)", title="Purple = tunnels (no GPS)"); ax.legend(fontsize=8); plt.show()

## 2 · The simplest filter: blend 🥤
Keep an estimate. When a GPS reading arrives, move a fraction `trust` of the way toward it.

### 🧩 Challenge 1 · Blend prediction and measurement

In [ ]:
def blend(estimate, measurement, trust):
    return ___      # 🧩 move part of the way toward the measurement

blend = check("blend", blend)

<details><summary>🤔 <b>Need a hint?</b></summary>

The surprise is <code>measurement − estimate</code>. Add a fraction of it.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>return estimate + trust * (measurement - estimate)      # 🧩 move part of the way toward the measurement</pre>

</details>

### 🎛️ Playground · How much should you trust the GPS?
This blend filter also predicts forward with the last velocity estimate. Try `trust = 1.0` (jumpy) and `trust = 0.02` (laggy). Is there one perfect value?

In [ ]:
def blend_filter(trust=0.3):
    est = np.zeros((T, 2)); pos, vel = gps[0], 8.0
    for t in range(T):
        if t: pos = pos + vel * dt                               # predict
        if has_gps[t]:
            new_pos = blend(pos, gps[t], trust)                  # correct the position
            if t:                                                # nudge speed toward the implied speed
                vel = blend(vel, (new_pos - est[t - 1, 0]) / dt, trust * 0.3)
            pos = new_pos
        est[t] = pos, vel
    err = np.sqrt(np.mean((est[has_gps, 0] - truth[has_gps, 0]) ** 2))
    tunnel_err = np.sqrt(np.mean((est[~has_gps, 0] - truth[~has_gps, 0]) ** 2))
    fig, ax = plt.subplots(figsize=(9, 3))
    road_plot(ax, 40, 140)
    ax.plot(time[40:140], est[40:140, 0], c="tab:orange", lw=2, label=f"blend filter (trust={trust})")
    ax.set(title=f"RMS error · with GPS: {err:.2f} m · in tunnels: {tunnel_err:.2f} m   (raw GPS: {gps_noise} m)", xlabel="seconds", ylabel="metres")
    ax.legend(fontsize=8); plt.show()

playground(blend_filter, trust=(0.02, 1.0, 0.02, 0.3))

## 3 · The Kalman filter: blending with automatic trust 🎯

The Kalman filter keeps **two** things: a best guess (`mean`) and how uncertain it is (`P`, a 2×2 *covariance* matrix). Each step:

| Step | In words | In maths |
|---|---|---|
| **Predict** | move the guess forward; uncertainty grows | `mean = A @ mean`, `P = A @ P @ A.T + Q` |
| **Correct** (if GPS) | compute trust **K** from uncertainties, then blend | `K = P[:,0] / (P[0,0] + R)`, `mean += K * surprise` |

### 🧩 Challenge 2 · The motion model
`A` turns *(position, velocity)* now into *(position, velocity)* one step later.

In [ ]:
def motion_model(dt):
    return np.array([[1.0, ___],      # 🧩 new position = position + ? × velocity
                     [0.0, 1.0]])      #    new velocity = velocity (the filter can't foresee the driver)

motion_model = check("motion_model", motion_model)
print(motion_model(0.25))

<details><summary>🤔 <b>Need a hint?</b></summary>

In <code>dt</code> seconds at velocity v you travel v × dt metres.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>return np.array([[1.0, dt],      # 🧩 new position = position + ? × velocity</pre>

</details>

### 🧩 Challenge 3 · How much to trust the GPS (the Kalman gain)

In [ ]:
def kalman_gain(P, R):
    return ___      # 🧩 my uncertainty ÷ (my uncertainty + GPS noise)

kalman_gain = check("gain", kalman_gain)

<details><summary>🤔 <b>Need a hint?</b></summary>

Numerator: <code>P[:, 0]</code>. Denominator: position uncertainty plus <code>R</code>.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>return P[:, 0] / (P[0, 0] + R)      # 🧩 my uncertainty ÷ (my uncertainty + GPS noise)</pre>

</details>

Here is the full filter. Every line maps onto the table above.

In [ ]:
def kalman(gps, has_gps, dt, jerk, R):
    A = motion_model(dt)
    g = np.array([dt**2 / 2, dt])
    Q = jerk**2 * np.outer(g, g) + 1e-6 * np.eye(2)       # how much uncertainty one step of driving adds
    mean, P = np.array([gps[0], 0.0]), np.diag([R, 100.0])  # start: near the first GPS, unsure of speed
    means, covs, pred_means, pred_covs = [], [], [], []
    for t in range(len(gps)):
        if t:
            mean, P = A @ mean, A @ P @ A.T + Q                 # PREDICT
        pred_means.append(mean); pred_covs.append(P)
        if has_gps[t]:
            K = kalman_gain(P, R)                                # how much to trust this reading
            surprise = gps[t] - mean[0]
            mean = mean + K * surprise                           # CORRECT the guess
            P = P - np.outer(K, P[0])                            # we're now more certain
        means.append(mean); covs.append(P)
    return np.array(means), np.array(covs), np.array(pred_means), np.array(pred_covs)

means, covs, pred_means, pred_covs = kalman(gps, has_gps, dt, jerkiness, gps_noise**2)
rms = lambda x, mask=slice(None): np.sqrt(np.mean((x[mask] - truth[mask, 0]) ** 2))
print("Compared on the SAME time steps (where GPS exists):")
print(f"  raw GPS        : {rms(gps, has_gps):.2f} m")
print(f"  Kalman filter  : {rms(means[:, 0], has_gps):.2f} m")
print(f"Inside tunnels (no GPS at all), Kalman keeps guessing: {rms(means[:, 0], ~has_gps):.2f} m")

In [ ]:
quiz("tunnel")

In [ ]:
sd = np.sqrt(covs[:, 0, 0])
fig, axs = plt.subplots(2, 1, figsize=(9, 5.5), sharex=True)
road_plot(axs[0], 40, 140)
axs[0].plot(time[40:140], means[40:140, 0], c="tab:blue", lw=2, label="Kalman estimate")
axs[0].fill_between(time[40:140], (means[:, 0] - 2 * sd)[40:140], (means[:, 0] + 2 * sd)[40:140], alpha=0.2, label="±2 std band")
axs[0].set(ylabel="metres", title="Zoom on the first tunnel"); axs[0].legend(fontsize=8)
axs[1].plot(time, sd); axs[1].set(xlabel="seconds", ylabel="position std (m)", title="Uncertainty over the whole drive")
for s, e in [(70, 110), (170, 185)]: axs[1].axvspan(time[s], time[e - 1], color="tab:purple", alpha=0.08)
plt.tight_layout(); plt.show()

## 4 · Hindsight: the smoother 🔙
After the drive is over, we can revise every estimate using **later** GPS readings too. This is the **Rauch–Tung–Striebel smoother**. It runs backwards over the filter's results.

In [ ]:
def smooth(means, covs, pred_means, pred_covs, dt):
    A = motion_model(dt)
    sm, sc = means.copy(), covs.copy()
    for t in range(len(means) - 2, -1, -1):                      # walk backwards in time
        J = covs[t] @ A.T @ np.linalg.inv(pred_covs[t + 1])        # how much the future should revise the past
        sm[t] = means[t] + J @ (sm[t + 1] - pred_means[t + 1])
        sc[t] = covs[t] + J @ (sc[t + 1] - pred_covs[t + 1]) @ J.T
    return sm, sc

smoothed, _ = smooth(means, covs, pred_means, pred_covs, dt)
print(f"RMS error · filter (online):    {rms(means[:, 0]):.2f} m")
print(f"RMS error · smoother (offline): {rms(smoothed[:, 0]):.2f} m")
fig, ax = plt.subplots(figsize=(9, 3))
road_plot(ax, 60, 120)
ax.plot(time[60:120], means[60:120, 0], label="filter (online)")
ax.plot(time[60:120], smoothed[60:120, 0], "--", lw=2, label="smoother (hindsight)")
ax.set(title="Inside the tunnel the smoother bridges the gap using where the car came out"); ax.legend(fontsize=8); plt.show()

In [ ]:
quiz("smoother")

## 5 · 🎛️ Playground · When the filter believes a wrong story
The filter needs a **noise model**. What if it *assumes* the GPS is better or worse than it really is? We count how often the truth lands inside the ±2 std band across 50 drives. A calibrated filter should score about **95%**. The true values are `gps_noise = 1.5` and `jerkiness = 0.6`.

In [ ]:
quiz("overconfident")

In [ ]:
def calibration(assumed_gps_noise=1.5, assumed_jerkiness=0.6):
    coverage = []
    for seed in range(50):                                   # 50 fresh drives, so one lucky drive can't mislead us
        tr, g, has = simulate_drive(seed)
        m, c, _, _ = kalman(g, has, dt, assumed_jerkiness, assumed_gps_noise**2)
        coverage.append(np.mean(np.abs(tr[:, 0] - m[:, 0]) <= 2 * np.sqrt(c[:, 0, 0])))
    m, c, _, _ = kalman(gps, has_gps, dt, assumed_jerkiness, assumed_gps_noise**2)
    s = np.sqrt(c[:, 0, 0])
    fig, ax = plt.subplots(figsize=(9, 2.8))
    road_plot(ax, 40, 140)
    ax.plot(time[40:140], m[40:140, 0], c="tab:blue"); ax.fill_between(time[40:140], (m[:, 0] - 2 * s)[40:140], (m[:, 0] + 2 * s)[40:140], alpha=0.2)
    ax.set(title=f"truth inside ±2 std band, averaged over 50 drives: {np.mean(coverage):.0%} (calibrated ≈ 95%)"); plt.show()

playground(calibration, assumed_gps_noise=(0.1, 6.0, 0.1, 1.5), assumed_jerkiness=(0.05, 3.0, 0.05, 0.6))

---
# Part B · Cameras and 3D 📷

## 6 · The pinhole camera
A camera at the origin looks along +Z. A 3D point `(X, Y, Z)` in camera coordinates lands on pixel `(u, v)`:

$$u = f\,\frac{X}{Z} + c_x \qquad v = f\,\frac{Y}{Z} + c_y$$

* `f`: focal length in pixels (zoom)
* `(cx, cy)`: the image centre

### 🧩 Challenge 4 · Project a 3D point to a pixel

In [ ]:
f, cx, cy = 250.0, 160.0, 120.0          # a 320 × 240 image

def project(X, Y, Z, f, cx, cy):
    u = ___                   # 🧩 horizontal pixel
    v = f * Y / Z + cy
    return u, v

project = check("project", project)

<details><summary>🤔 <b>Need a hint?</b></summary>

Exactly like the v line below it, but with X and cx.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>u = f * X / Z + cx                   # 🧩 horizontal pixel</pre>

</details>

### 🎛️ Playground · Move the camera
A wireframe box of 3D points sits in front of the camera. Slide the camera sideways and backwards, and watch the image change. Near corners move more than far corners, which is **parallax**, the cue your brain uses for depth.

In [ ]:
corners = np.array([[x, y, z] for x in (-1, 1) for y in (-0.7, 0.7) for z in (4, 6)], float)
edges = [(i, j) for i in range(8) for j in range(i + 1, 8) if np.sum(corners[i] != corners[j]) == 1]

def camera_view(camera_x=0.0, camera_back=0.0):
    cam = corners - np.array([camera_x, 0.0, -camera_back])      # world → camera coordinates (camera only translates)
    u, v = project(cam[:, 0], cam[:, 1], cam[:, 2], f, cx, cy)
    fig, ax = plt.subplots(figsize=(4.8, 3.6))
    for i, j in edges:
        near = cam[i, 2] < 5 + camera_back and cam[j, 2] < 5 + camera_back
        ax.plot([u[i], u[j]], [v[i], v[j]], c="tab:red" if near else "tab:blue", lw=2)
    ax.set(xlim=(0, 320), ylim=(240, 0), title="camera image (red = near face)", xlabel="u (pixels)", ylabel="v (pixels)")
    ax.set_aspect("equal"); plt.show()

playground(camera_view, camera_x=(-2.0, 2.0, 0.1, 0.0), camera_back=(0.0, 6.0, 0.25, 0.0))

In [ ]:
quiz("depth")

In [ ]:
X, Y, Z = 0.8, -0.3, 3.0
print("point at distance 3  → pixel", np.round(project(X, Y, Z, f, cx, cy), 3))
print("point at distance 6  → pixel", np.round(project(2 * X, 2 * Y, 2 * Z, f, cx, cy), 3))
print("A single image cannot tell these apart. Depth is lost.")

## 7 · Getting 3D back with depth 🧊
If a sensor (LiDAR, stereo, or a depth-predicting network) also tells us **Z** for each pixel, we can invert the projection.

### 🧩 Challenge 5 · Back to 3D using depth

In [ ]:
def unproject(u, v, Z, f, cx, cy):
    X = ___                 # 🧩 undo the projection
    Y = (v - cy) * Z / f
    return X, Y, Z

unproject = check("unproject", unproject)

u, v = project(corners[:, 0], corners[:, 1], corners[:, 2], f, cx, cy)
Xr, Yr, Zr = unproject(u, v, corners[:, 2], f, cx, cy)
print("max 3D reconstruction error:", np.abs(np.c_[Xr, Yr, Zr] - corners).max())

<details><summary>🤔 <b>Need a hint?</b></summary>

Mirror the Y line: subtract cx, multiply by Z, divide by f.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>X = (u - cx) * Z / f                 # 🧩 undo the projection</pre>

</details>

### 🎛️ Playground · Noisy depth
Real depth sensors get **worse with distance** (stereo error grows roughly with distance²). Push the scene further away and add noise. Where does 3D reconstruction break down?

In [ ]:
grid = np.array([[x, y] for x in np.linspace(-1.5, 1.5, 7) for y in np.linspace(-1, 1, 5)])
def noisy_depth(distance=4.0, depth_noise=0.01):
    r = np.random.default_rng(0)
    pts = np.c_[grid, np.full(len(grid), distance)]
    u, v = project(pts[:, 0], pts[:, 1], pts[:, 2], f, cx, cy)
    Zm = pts[:, 2] + r.normal(0, depth_noise * distance**2, len(pts))   # noise grows with distance²
    rec = np.stack(unproject(u, v, Zm, f, cx, cy), axis=1)             # (points, 3)
    err = np.linalg.norm(rec - pts, axis=1).mean()
    fig, ax = plt.subplots(figsize=(5, 3.2))
    ax.scatter(pts[:, 0], pts[:, 2], label="true (top view)"); ax.scatter(rec[:, 0], rec[:, 2], marker="x", label="reconstructed")
    ax.set(xlabel="X (m)", ylabel="Z = depth (m)", title=f"mean 3D error {err:.3f} m"); ax.legend(fontsize=8); plt.show()

playground(noisy_depth, distance=(1.0, 15.0, 0.5, 4.0), depth_noise=(0.0, 0.05, 0.002, 0.01))

---
## 8 · Recap and industry links 🏭

| You built | Where it runs |
|---|---|
| Blend + Kalman filter | every phone's location, drone autopilots, Waymo/Tesla sensor fusion, Apple Vision Pro head tracking |
| Uncertainty that grows without data | robots deciding when to slow down or re-localise |
| Smoother (hindsight) | offline map building, auto-labelling training data |
| Calibration check | safety cases for autonomous systems |
| Pinhole projection and un-projection | every robot camera; the first line of **NeRF** and **3D Gaussian Splatting** renderers (NVIDIA Instant-NGP, Meta scene reconstruction) |

**Deep Kalman filters and world models** (lecture 11) learn `A`, `Q` and the observation model from data instead of writing them by hand. The *predict → correct* loop you just built stays the same.

### 🧪 HW1 Part A, beginner version
1. Make the second tunnel 60 steps long. Plot filter error inside vs outside tunnels.
2. In the calibration playground, make the assumed GPS noise much *larger* than 1.5. Coverage goes up. Is that good? (Look at the band width.)
3. Why must you never report the smoother's accuracy as if it were a real-time result?

### 🗣️ Explain it back
In one sentence each: what does `Q` mean, and what does `R` mean?

In [ ]:
progress_report()